In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from glob import glob
# Aplicar configuraciones de visualización total de Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

from pathlib import Path
import glob

import warnings
warnings.filterwarnings("ignore")
import plotly.express as px

import func_negocio as func

In [7]:
df = pd.read_excel("Recalculos de dotacion - Cadena - CorridaJunio 3.xlsx")

In [8]:
def graficar_dispersion_formato(df: pd.DataFrame, area: str, leyenda: str) -> None:
    sns.set_theme(style="whitegrid")
    
    filtro_filas = (df["Área"] == area) & (df["GOS"] != "Mini")
    columnas_necesarias = ["JEq", "VtaNeta", leyenda]
    df_filtrado = df.loc[filtro_filas, columnas_necesarias].dropna()
    
    g = sns.JointGrid(
        data=df_filtrado, 
        x="VtaNeta", 
        y="JEq", 
        hue=leyenda, 
        palette="tab10", 
        height=7, 
        ratio=5
    )
    
    g.plot_joint(
        sns.scatterplot, 
        alpha=0.8, 
        edgecolor="w", 
        linewidth=0.7, 
        s=80
    )
    
    g.plot_marginals(
        sns.histplot, 
        kde=False, 
        alpha=0.6, 
        multiple="stack"
    )
    
    # --- Ajuste y cálculo de la Regresión Lineal Global ---
    x_vals = df_filtrado["VtaNeta"].to_numpy()
    y_vals = df_filtrado["JEq"].to_numpy()
    
    # Ajuste polinomial de grado 1 (y = mx + b)
    pendiente, intercepto = np.polyfit(x_vals, y_vals, 1)
    
    # Cálculo eficiente de R² usando operaciones vectorizadas de NumPy
    y_pred = pendiente * x_vals + intercepto
    ss_res = np.sum((y_vals - y_pred) ** 2)
    ss_tot = np.sum((y_vals - np.mean(y_vals)) ** 2)
    r_cuadrado = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0.0
    
    # Generar puntos sobre el eje X para trazar la línea continua
    x_reg = np.linspace(x_vals.min(), x_vals.max(), 100)
    y_reg = pendiente * x_reg + intercepto
    
    # Dibujar la recta sobre el eje joint central
    g.ax_joint.plot(
        x_reg, 
        y_reg, 
        color="#d95f02", 
        linestyle="--", 
        linewidth=2, 
        label="Regresión Global"
    )
    
    # Construcción de la cadena con la ecuación e incluyendo R²
    texto_ecuacion = f"y = {pendiente:.2e}x + {intercepto:.2e}\n$R^2$ = {r_cuadrado:.3f}"
    
    # Inserción de las métricas en la esquina superior izquierda
    g.ax_joint.text(
        0.05, 0.95, 
        texto_ecuacion, 
        transform=g.ax_joint.transAxes, 
        fontsize=11, 
        fontweight="bold", 
        color="#d95f02",
        verticalalignment="top",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="#d95f02", alpha=0.9)
    )
    
    # Configuración de etiquetas y títulos
    g.ax_joint.set_xlabel("Venta Neta (Moneda)", fontsize=11)
    g.ax_joint.set_ylabel("JEq", fontsize=11)
    
    g.fig.suptitle(
        f"Análisis de Dispersión, Distribución y Regresión (Área: {area})", 
        fontsize=14, 
        fontweight="bold", 
        y=1.02
    )
    
    g.ax_joint.legend(title=leyenda, title_fontsize=11, fontsize=10)
    
    plt.show()

In [9]:
df["Área"].unique()

array(['TOTAL', 'ABARROTES', 'ALMACÉN', 'ALMACÉN FRESCOS', 'CAJAS',
       'ELECTRO', 'FRESCOS', 'INVENTARIOS', 'MULTIFUNCIONAL', 'RECEPCIÓN',
       'SIN ELECTRO'], dtype=object)

In [10]:
df["Formato"].unique()

array(['HIPER', 'SUPER', 'EXPRESS', 'VIVANDA'], dtype=object)

In [11]:
df["ratio"] = df["CAPE"] / df["JEq"]
df.groupby("Área")["ratio"].mean()

Área
ABARROTES          1.087304
ALMACÉN            1.136294
ALMACÉN FRESCOS    1.156985
CAJAS              1.146418
ELECTRO            1.148076
FRESCOS            1.149969
INVENTARIOS        1.175445
MULTIFUNCIONAL     1.147208
RECEPCIÓN          1.122666
SIN ELECTRO             NaN
TOTAL                   NaN
Name: ratio, dtype: float64

In [12]:
df[df["GOS"]=="Sofía V"]["Tienda"].unique().tolist()

['1198 Cusco Wanchaq - PVH',
 '2638 SJB Ayacucho - PVH',
 'P075 Trujillo - PVH',
 'P102 Chiclayo - PVH',
 'P104 Arequipa - PVH',
 'P115 Huancayo - PVH',
 'P122 Ica - PVH',
 'P123 El Chacarero - PVH',
 'P130 Chimbote - PVH',
 'P142 Chincha - PVH',
 'P143 Nvo Chimbote - PVH',
 'P144 Huacho - PVH',
 'P145 Piura - PVH',
 'P146 Tacna - PVH',
 'P147 Juliaca - PVH',
 'P172 Puno - PVH',
 'P177 Talara - PVH',
 'P192 Sullana - PVH',
 'P201 Huaral - PVH',
 'P221 Barranca - PVH',
 'P223 Cusco Mall - PVH',
 'P225 Cajamarca - PVH',
 'P226 Paita - PVH',
 'P236 Pisco - PVH',
 'P259 Moquegua - PVH',
 'P262 Talara Municipalidad - PVH',
 'P516 Ilo - PVH',
 'P724 Tumbes - PVH',
 'P769 Chiclayo Aventura - PVH',
 'SO02 Pucallpa - PVO',
 'SO03 Huanuco - PVO',
 'SO04 Jaen - PVO',
 'SO05 Tarapoto - PVO',
 'SO06 Tarapoto 2 - PVO']

In [13]:
df["Region"] = df["cluster"].str[0:3]
df["formato2"] = df["Tienda"].str[-3:]
from sklearn.linear_model import LinearRegression

In [18]:

def calcular_capo (df, tienda):
    area = "ABARROTES"
    from sklearn.linear_model import LinearRegression
    region = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["Region"].to_numpy()[0]
    cluster = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["cluster"].to_numpy()[0]
    formato = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["formato2"].to_numpy()[0]
    promedio_venta = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["VtaNeta"].mean()
    lista_tiendas = df[(df["Área"]==area)&(df["cluster"]==cluster)&(df["Region"]==region)&((df["VtaNeta"]>promedio_venta*0.9)&(df["VtaNeta"]<promedio_venta*1.1))]["Tienda"].unique().tolist()
    print(area)
    print(promedio_venta)
    print(lista_tiendas)
    df_filtrado = df[(df["Tienda"].isin(lista_tiendas))&(df["Área"]==area)&(df["Periodo"]!=202603)][["VtaNeta", "JEq"]].dropna()
    x_raw = df_filtrado["VtaNeta"].to_numpy()
    y_vals = df_filtrado["JEq"].to_numpy()

    x_vals = x_raw.reshape(-1, 1)
    modelo = LinearRegression()
    modelo.fit(x_vals, y_vals)
    pendiente = modelo.coef_[0]
    intercepto = modelo.intercept_
    r_cuadrado = modelo.score(x_vals, y_vals)


    dff = df[(df["Tienda"]==tienda)&(df["Área"]==area)&(df["Periodo"]!=202603)]
    x_nuevos = dff["VtaNeta"].to_numpy().reshape(-1, 1)
    dff["JeqModelo"] = modelo.predict(x_nuevos)
    dff["JeqModelo"] = np.ceil(dff["JeqModelo"] * 2) / 2
    dff["CapoPropuesto"] = dff["JeqModelo"]*1.10


    df1 = dff[["Tienda", "Área", "CapoPropuesto"]]

    area = "FRESCOS"

    region = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["Region"].to_numpy()[0]
    cluster = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["cluster"].to_numpy()[0]
    formato = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["formato2"].to_numpy()[0]
    promedio_venta = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["VtaNeta"].mean()

    lista_tiendas = df[(df["Área"]==area)&(df["cluster"]==cluster)&(df["Region"]==region)&((df["VtaNeta"]>promedio_venta*0.9)&(df["VtaNeta"]<promedio_venta*1.1))]["Tienda"].unique().tolist()
    print(area)
    print(promedio_venta)
    print(lista_tiendas)

    df_filtrado = df[(df["Tienda"].isin(lista_tiendas))&(df["Área"]==area)][["VtaNeta", "JEq"]].dropna()
    print(df_filtrado)
    x_raw = df_filtrado["VtaNeta"].to_numpy()
    y_vals = df_filtrado["JEq"].to_numpy()

    x_vals = x_raw.reshape(-1, 1)
    modelo = LinearRegression()
    modelo.fit(x_vals, y_vals)
    pendiente = modelo.coef_[0]
    intercepto = modelo.intercept_
    r_cuadrado = modelo.score(x_vals, y_vals)


    dff = df[(df["Tienda"]==tienda)&(df["Área"]==area)]
    x_nuevos = dff["VtaNeta"].to_numpy().reshape(-1, 1)
    dff["JeqModelo"] = modelo.predict(x_nuevos)
    dff["JeqModelo"] = np.ceil(dff["JeqModelo"] * 2) / 2
    dff["CapoPropuesto"] = dff["JeqModelo"]*1.15


    df2 = dff[["Tienda", "Área", "CapoPropuesto"]]

    area = "ALMACÉN"
    from sklearn.linear_model import LinearRegression
    region = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["Region"].to_numpy()[0]
    cluster = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["cluster"].to_numpy()[0]
    formato = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["formato2"].to_numpy()[0]
    promedio_venta = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["VtaNeta"].mean()

    lista_tiendas = df[(df["Área"]==area)&(df["cluster"]==cluster)&(df["Region"]==region)&((df["VtaNeta"]>promedio_venta*0.9)&(df["VtaNeta"]<promedio_venta*1.1))]["Tienda"].unique().tolist()
    print(area)
    print(promedio_venta)
    print(lista_tiendas)
    df_filtrado = df[(df["Tienda"].isin(lista_tiendas))&(df["Área"]==area)][["VtaNeta", "JEq"]].dropna()
    x_raw = df_filtrado["VtaNeta"].to_numpy()
    y_vals = df_filtrado["JEq"].to_numpy()

    x_vals = x_raw.reshape(-1, 1)
    modelo = LinearRegression()
    modelo.fit(x_vals, y_vals)
    pendiente = modelo.coef_[0]
    intercepto = modelo.intercept_
    r_cuadrado = modelo.score(x_vals, y_vals)



    dff = df[(df["Tienda"]==tienda)&(df["Área"]==area)]
    x_nuevos = dff["VtaNeta"].to_numpy().reshape(-1, 1)
    dff["JeqModelo"] = modelo.predict(x_nuevos)
    dff["JeqModelo"] = np.ceil(dff["JeqModelo"] * 2) / 2
    dff["CapoPropuesto"] = dff["JeqModelo"]*1.13


    df3 = dff[["Tienda", "Área", "CapoPropuesto"]]

    area = "ALMACÉN FRESCOS"
    from sklearn.linear_model import LinearRegression
    region = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["Region"].to_numpy()[0]
    cluster = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["cluster"].to_numpy()[0]
    formato = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["formato2"].to_numpy()[0]
    promedio_venta = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["VtaNeta"].mean()

    lista_tiendas = df[(df["Área"]==area)&(df["cluster"]==cluster)&(df["Region"]==region)&((df["VtaNeta"]>promedio_venta*0.9)&(df["VtaNeta"]<promedio_venta*1.1))]["Tienda"].unique().tolist()
    print(area)
    print(promedio_venta)
    print(lista_tiendas)
    df_filtrado = df[(df["Tienda"].isin(lista_tiendas))&(df["Área"]==area)][["VtaNeta", "JEq"]].dropna()
    x_raw = df_filtrado["VtaNeta"].to_numpy()
    y_vals = df_filtrado["JEq"].to_numpy()

    x_vals = x_raw.reshape(-1, 1)
    modelo = LinearRegression()
    modelo.fit(x_vals, y_vals)
    pendiente = modelo.coef_[0]
    intercepto = modelo.intercept_
    r_cuadrado = modelo.score(x_vals, y_vals)



    dff = df[(df["Tienda"]==tienda)&(df["Área"]==area)]
    x_nuevos = dff["VtaNeta"].to_numpy().reshape(-1, 1)
    dff["JeqModelo"] = modelo.predict(x_nuevos)
    dff["JeqModelo"] = np.ceil(dff["JeqModelo"] * 2) / 2
    dff["CapoPropuesto"] = dff["JeqModelo"]*1.13


    df4 = dff[["Tienda", "Área", "CapoPropuesto"]]

    area = "CAJAS"
    from sklearn.linear_model import LinearRegression
    region = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["Region"].to_numpy()[0]
    cluster = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["cluster"].to_numpy()[0]
    formato = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["formato2"].to_numpy()[0]
    promedio_venta = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["VtaNeta"].mean()

    lista_tiendas = df[(df["Área"]==area)&(df["cluster"]==cluster)&(df["Region"]==region)&((df["VtaNeta"]>promedio_venta*0.9)&(df["VtaNeta"]<promedio_venta*1.1))]["Tienda"].unique().tolist()
    print(area)
    print(promedio_venta)
    print(lista_tiendas)

    df_filtrado = df[(df["Tienda"].isin(lista_tiendas))&(df["Área"]==area)][["VtaNeta", "JEq"]].dropna()
    x_raw = df_filtrado["VtaNeta"].to_numpy()
    y_vals = df_filtrado["JEq"].to_numpy()

    x_vals = x_raw.reshape(-1, 1)
    modelo = LinearRegression()
    modelo.fit(x_vals, y_vals)
    pendiente = modelo.coef_[0]
    intercepto = modelo.intercept_
    r_cuadrado = modelo.score(x_vals, y_vals)



    dff = df[(df["Tienda"]==tienda)&(df["Área"]==area)]
    x_nuevos = dff["VtaNeta"].to_numpy().reshape(-1, 1)
    dff["JeqModelo"] = modelo.predict(x_nuevos)
    dff["JeqModelo"] = np.ceil(dff["JeqModelo"] * 2) / 2
    dff["CapoPropuesto"] = dff["JeqModelo"]*1.14


    df5 = dff[["Tienda", "Área", "CapoPropuesto"]]

    area = "RECEPCIÓN"
    from sklearn.linear_model import LinearRegression
    region = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["Region"].to_numpy()[0]
    cluster = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["cluster"].to_numpy()[0]
    formato = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["formato2"].to_numpy()[0]
    promedio_venta = df[(df["Tienda"]==tienda)&(df["Área"]==area)]["VtaNeta"].mean()

    lista_tiendas = df[(df["Área"]==area)&(df["cluster"]==cluster)&(df["Region"]==region)&((df["VtaNeta"]>promedio_venta*0.9)&(df["VtaNeta"]<promedio_venta*1.1))]["Tienda"].unique().tolist()
    print(area)
    print(promedio_venta)
    print(lista_tiendas)

    df_filtrado = df[(df["Tienda"].isin(lista_tiendas))&(df["Área"]==area)][["VtaNeta", "JEq"]].dropna()
    x_raw = df_filtrado["VtaNeta"].to_numpy()
    y_vals = df_filtrado["JEq"].to_numpy()

    x_vals = x_raw.reshape(-1, 1)
    modelo = LinearRegression()
    modelo.fit(x_vals, y_vals)
    pendiente = modelo.coef_[0]
    intercepto = modelo.intercept_
    r_cuadrado = modelo.score(x_vals, y_vals)



    dff = df[(df["Tienda"]==tienda)&(df["Área"]==area)]
    x_nuevos = dff["VtaNeta"].to_numpy().reshape(-1, 1)
    dff["JeqModelo"] = modelo.predict(x_nuevos)
    dff["JeqModelo"] = np.ceil(dff["JeqModelo"] * 2) / 2
    dff["CapoPropuesto"] = dff["JeqModelo"]*1.12
    dff["Diff"] = dff["JeqModelo"] - dff["JEq"]


    df6 = dff[["Tienda", "Área", "CapoPropuesto"]]

    df_final = pd.concat([df1, df2, df3, df4, df5, df6])

    df_final1 = df_final.groupby(["Tienda", "Área"])["CapoPropuesto"].median().reset_index()
    df_final1["CapoPropuesto"] = np.ceil(df_final1["CapoPropuesto"] * 2) / 2
    return df_final1

In [19]:
tienda = 'P226 Paita - PVH'
calcular_capo(df, tienda)

ABARROTES
3943544.5768055427
['P130 Chimbote - PVH', 'P143 Nvo Chimbote - PVH', 'P144 Huacho - PVH', 'P172 Puno - PVH', 'P226 Paita - PVH', 'P236 Pisco - PVH', 'P259 Moquegua - PVH', 'P262 Talara Municipalidad - PVH', 'P516 Ilo - PVH', 'P724 Tumbes - PVH', 'SO04 Jaen - PVO', 'SO06 Tarapoto 2 - PVO']
FRESCOS
743143.6959234047
['P143 Nvo Chimbote - PVH', 'P225 Cajamarca - PVH', 'P226 Paita - PVH', 'P236 Pisco - PVH', 'P259 Moquegua - PVH', 'P516 Ilo - PVH', 'P724 Tumbes - PVH', 'SO03 Huanuco - PVO', 'SO04 Jaen - PVO']
            VtaNeta        JEq
3174  635680.400000   9.677923
3185  610552.480000   8.144948
3196  689699.900000  10.795759
3207  724690.390000  10.830167
3218  689422.150000  10.151882
3229  649300.550896  10.750104
5220  791860.790000   7.634704
5231  811803.840000   8.241458
5242  937042.430000   9.279469
5253  865135.930000   8.797931
5264  960126.770000   9.958253
5275  867009.893038   9.833736
5286  674568.230000   8.359966
5297  693753.650000   7.461146
5308  835999.

,Tienda,Área,CapoPropuesto
0,P226 Paita - PVH,ABARROTES,14.5
1,P226 Paita - PVH,ALMACÉN,8.0
2,P226 Paita - PVH,ALMACÉN FRESCOS,5.0
3,P226 Paita - PVH,CAJAS,20.0
4,P226 Paita - PVH,FRESCOS,11.5
5,P226 Paita - PVH,RECEPCIÓN,3.0


In [16]:
# lista_tiendas = [
# '1198 Cusco Wanchaq - PVH',
# #  '2638 SJB Ayacucho - PVH',
# #  'P075 Trujillo - PVH',
#  'P102 Chiclayo - PVH',
#  'P104 Arequipa - PVH',
#  'P115 Huancayo - PVH',
#  'P122 Ica - PVH',
#  'P123 El Chacarero - PVH',
#  'P130 Chimbote - PVH',
#  'P142 Chincha - PVH',
#  'P143 Nvo Chimbote - PVH',
#  'P144 Huacho - PVH',
#  'P145 Piura - PVH',
#  'P146 Tacna - PVH',
#  'P147 Juliaca - PVH',
#  'P172 Puno - PVH',
#  'P177 Talara - PVH',
#  'P192 Sullana - PVH',
#  'P201 Huaral - PVH',
# #  'P221 Barranca - PVH',
#  'P223 Cusco Mall - PVH',
#  'P225 Cajamarca - PVH',
#  'P226 Paita - PVH',
#  'P236 Pisco - PVH',
#  'P259 Moquegua - PVH',
#  'P262 Talara Municipalidad - PVH',
#  'P516 Ilo - PVH',
#  'P724 Tumbes - PVH',
#  'P769 Chiclayo Aventura - PVH',
#  'SO02 Pucallpa - PVO',
#  'SO03 Huanuco - PVO',
#  'SO04 Jaen - PVO',
#  'SO05 Tarapoto - PVO',
#  'SO06 Tarapoto 2 - PVO']

# acumulador = []

# for x in lista_tiendas:
#     print(x)
#     df_tienda = calcular_capo(df, x)
#     acumulador.append(df_tienda)

# # Se consolida todo en el DataFrame final en un solo paso
# df_acumulado = pd.concat(acumulador, ignore_index=True)


In [17]:
df_acumulado.to_excel("revisar2.xlsx", index=False)

NameError: name 'df_acumulado' is not defined